Ideia da implementação: Sistema de registro de carros a venda para o Brasil

In [29]:
%pip install -q poetry pydantic


Note: you may need to restart the kernel to use updated packages.


In [30]:
# Para não precisar de trocar o kernel para usar poetry
%pip install -q "pydantic[email]==2.6.1" fastapi httpx


Note: you may need to restart the kernel to use updated packages.


In [31]:
# Import geral
from enum import auto, IntFlag, IntEnum
from datetime import datetime
from typing import Any
from uuid import uuid4
import hashlib
import re

from pydantic import (
    BaseModel,
    EmailStr,
    SecretStr,
    ValidationError,
    UUID4,
    Field,
    field_validator,
    model_validator,
    field_serializer,
    model_serializer,
)

VALID_NAME_REGEX = re.compile(r"^[a-zA-ZÀ-ÿ\s]{5,}$") # realmente aceitando todos caracteres do alfabeto latino
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")


In [32]:
class Cargo(IntEnum): # Um ou outro, por isso IntEnum
    Cliente = 0 # Apenas compra, estrela, etc
    Vendedor = 1 # Permite colocar carros a venda, alterar preços, promoções, etc
    Admin = 2 # Editar cargos, e tudo que o vendedor pode


In [33]:
class Usuario(BaseModel):
    id: UUID4 = Field(
        default_factory=uuid4, 
        description="ID único do usuário", 
        kw_only=True,
    )
    nome: str = Field(
        examples=["Júlio Carvalho Cavalcanti", "Cauã"], # https://pt.fakenamegenerator.com/gen-male-br-br.php
        description="Nome do usuário",
    )
    telefone: str = Field(
        examples=["(62) 91111-5555","62788888888"], # Padronizar manualmente para dar mais liberadade ao usuário
        description=["Telefone de comunicação com o usuário"]
    )
    email: EmailStr = Field(
        examples=["jcarvalho@gmail.com", "caua5562@gmail.com"],
        description=["Email de comunicação do usuário"],
    )
    senha: SecretStr = Field(
        examples=["SEnha123"],
        description=["A senha que o usuário decidiu (ilegível diretamente)"],
        exclude=True,
    )
    cargo: Cargo = Field(
        default=Cargo.Cliente,
        description=["Cargo do usuário no sistema"]
    )

    # Validações
    @field_validator("nome")
    @classmethod
    def validate_nome(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Nome inválido! O nome precisa de ter apenas letras e pelo menos 5 delas."
            )
        return v
    
    @field_validator("cargo", mode="before") # Improvável, tem um default
    @classmethod
    def validate_cargo(cls, v: int | str | Cargo) -> Cargo:
        op = {int: lambda x: Cargo(x), str: lambda x: Cargo[x], Cargo: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f'Cargo inválido! Contate o técnico de TI.'
            )
    
    @field_validator("telefone")
    @classmethod
    def validate_telefone(cls, v: str) -> str:
        # Não existe DDD de 3 dígitos no brasil (https://simcompany.com.br/ddd-do-brasil-por-estado-guia-para-consulta/)
        # Então 2 digitos DDD + 8 digitos (+ 1 digito opcional) = 10~11
        # Assume que a interface vai informar para o Usuário que é +55 e que é apenas para o Brasil
        numero_puro = re.sub(r"\D", "", v)
        if len(numero_puro) not in (10, 11):
            raise ValueError(
                "Telefone inválido! O número precisa ter 10 ou 11 dígitos."
            )
        return numero_puro  # Salva sem máscara
    
    @model_validator(mode="before")
    @classmethod
    def validate_usuario(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "nome" not in v or "email" not in v or "senha" not in v: # Garantir presença dados básicos
            raise ValueError("Dados incompletos!")
        if not VALID_PASSWORD_REGEX.match(v["senha"]): # Validar senha pelo REGEX
            raise ValueError(
                "Senha fraca! A senha precisa de: Pelo menos 8 caracteres, números e letras maiusculas e minusculas"
            )
        v["senha"] = hashlib.sha256(v["senha"].encode()).hexdigest() # Criptografia da senha
        return v
    
    @field_serializer("cargo", when_used="json")
    @classmethod
    def serialize_cargo(cls, v) -> str:
        return v.name # Retornar o nome, não o valor int

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)

    @model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        # info é um filtro simples, se nenhum estiver presente vai só nome e cargo
        if not info.include and not info.exclude:
            return {"nome": self.nome, "cargo": self.cargo.name}
        return serializer(self) # serialização do pydantic


In [34]:
class Carro(BaseModel):
    id: UUID4 = Field(
        default_factory=uuid4, 
        description="ID único do carro", 
        kw_only=True,
    )
    # Tirei os exemplos do exemplo do Gemini depois da pesquisa no google
    marca: str = Field(
        examples=["Fiat", "Volkswagen", "Chevrolet", "Toyota", "Hyundai"],
        description="Marca do carro",
    )
    modelo: str = Field(
        examples=["Onix", "HB20", "VW Polo"],
        description="Modelo do carro",
    )
    ano: int = Field(
        examples=[2013, 2020, 2022],
        description="Ano de fabricação do carro",
    )
    preco: float = Field(
        examples=[50000.00, 120000.00],
        description="Preço de venda em reais",
    )
    cor: str = Field(
        examples=["Preto", "Branco", "Prata"],
        description="Cor do carro",
    )
    vendedor_id: UUID4 = Field(
        description="ID do vendedor",
    )

    @field_validator("ano")
    @classmethod
    def validate_ano(cls, v: int) -> int:
        #https://www.google.com/search?client=opera&q=primeiro+carro&sourceid=opera&ie=UTF-8&oe=UTF-8#:~:text=O%20primeiro%20autom%C3%B3vel%20movido%20a%20gasolina%20foi%20o%20Benz%20Patent%2DMotorwagen%2C%20patenteado%20por%20Karl%20Benz%20em%2029%20de%20janeiro%20de%201886%20na%20Alemanha.
        # 1º Carro é alemão e de 1886
        ano_atual = int(datetime.now().year)
        if (1886 > v) or (v > ano_atual):
            raise ValueError(f"Ano inválido! O ano deve ser entre 1886 e {ano_atual}.")
        return v

    @field_validator("preco")
    @classmethod
    def validate_preco(cls, v: float) -> float:
        if v <= 0: # Nenhum carro de graça
            raise ValueError("Preço inválido! O preço deve ser maior que zero.")
        return v

    # Sem validação de marca, modelo e cor por serem gerais demais, ao ponto que provavelmente seria um dropdown (ou similar) e por isso não faz tanto sentido validação extra

    @model_validator(mode="before")
    @classmethod
    def validate_carro(cls, v: dict[str, Any]) -> dict[str, Any]:
        # Deixando mais granular
        if "marca" not in v or "modelo" not in v or "cor" not in v: # Check basico
            raise ValueError("Dados do carro estão incompletos! Verifique a marca, modelo ou cor.")

        if "ano" not in v:
            raise ValueError("Ano de criação do carro não foi informado!")
        
        if "preco" not in v:
            raise ValueError("Valor do carro não foi informado!")

        if "vendedor_id" not in v:
            raise ValueError("Produto sem vendedor! Informe o técnico de TI.")
        
        return v

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)

    @model_serializer(mode="wrap", when_used="json")
    def serialize_carro(self, serializer, info) -> dict[str, Any]:
        # info como filtro, se nenhum for requisitado mandar os dados mínimos
        if not info.include and not info.exclude:
            return {"marca": self.marca, "modelo": self.modelo, "cor": self.cor}
        return serializer(self)


In [35]:
# Criando uma função só para deixar mais fácil a leitura, baseada na validate()
def validar(tipo: BaseModel, data: dict[str, Any]) -> None:
    try:
        objeto = tipo.model_validate(data)
        print(objeto)
    except ValidationError as e:
        print("Objeto inválido") # Função é de debug, por isso esse print horrendo está ok
        for error in e.errors():
            print(error)


In [36]:
def main() -> None:
    dict_usuarios = dict(
        # Cargo padrão é cliente
        cliente_1={
            "nome": "Júlio Carvalho Cavalcanti",
            "telefone": "(62) 91111-5555",
            "email": "jcarvalho@gmail.com",
            "senha": "Senha123",
        },
        cliente_2={
            "nome": "Cauã Silva",
            "telefone": "62788888888",
            "email": "caua5562@gmail.com",
            "senha": "Senha456",
        },
        vendedor_1={
            "nome": "Pedro Augusto",
            "telefone": "(11) 98888-7777",
            "email": "pedro.augusto@gmail.com",
            "senha": "Vend3dor123",
            "cargo": "Vendedor",
        },
    )

    vendedor_id = uuid4()

    dict_carros = dict(
        carro_bom_1={
            "marca": "Fiat",
            "modelo": "Onix",
            "ano": 2020,
            "preco": 65000.00,
            "cor": "Preto",
            "vendedor_id": vendedor_id,
        },
        carro_bom_2={
            "marca": "Toyota",
            "modelo": "Corolla",
            "ano": 2022,
            "preco": 130000.00,
            "cor": "Branco",
            "vendedor_id": vendedor_id,
        },
        carro_ano_invalido={
            "marca": "Volkswagen",
            "modelo": "Polo",
            "ano": 1800,  # Antigo demais
            "preco": 90000.00,
            "cor": "Prata",
            "vendedor_id": vendedor_id,
        },
        carro_preco_invalido={
            "marca": "Honda",
            "modelo": "Civic",
            "ano": 2019,
            "preco": -5000.00,  # Preço negativo
            "cor": "Azul",
            "vendedor_id": vendedor_id,
        },
        carro_dados_faltando={ # carro_bom_1 sem modelo
            "marca": "Fiat",
            "ano": 2020,
            "preco": 65000.00,
            "cor": "Preto",
            "vendedor_id": vendedor_id,
        },
        carro_sem_vendedor={ # carro_bom_2 sem vendedor
            "marca": "Toyota",
            "modelo": "Corolla",
            "ano": 2022,
            "preco": 130000.00,
            "cor": "Branco",
        },
    )

    print("-"*25," USUARIOS ","-"*25)
    for nome, data in dict_usuarios.items():
        print(nome)
        validar(Usuario, data)

    print("-"*25," CARROS ","-"*25)
    for nome, data in dict_carros.items():
        print(nome)
        validar(Carro, data)

    # Para demonstrar como ficaria em um BD
    print("-"*25," USUARIO JSON ","-"*25)
    vendedor_json={ # Tirei do dict para não passar pelo loop, porque se não a senha é criptografada e por isso é 'invalida'
            "nome": "Julian Carvalho Castro",
            "telefone": "(61 98883-7577", # Numero 'errado' mas a validação lida com isso sem problemas
            "email": "julian.castro@gmail.com",
            "senha": "Vendedor2",
            "cargo": "Vendedor",
        }
    vendedor = Usuario.model_validate(vendedor_json)
    print(vendedor.model_dump())

    print("-"*25," CARRO JSON ","-"*25)
    carro = Carro.model_validate(dict_carros['carro_bom_1'])
    print(carro.model_dump())


In [37]:
if __name__ == '__main__':
    main()


-------------------------  USUARIOS  -------------------------
cliente_1
id=UUID('72b0f37d-15bc-461f-a638-60ac6649bd68') nome='Júlio Carvalho Cavalcanti' telefone='62911115555' email='jcarvalho@gmail.com' senha=SecretStr('**********') cargo=<Cargo.Cliente: 0>
cliente_2
id=UUID('bf32d987-c94e-4f50-bb54-3c1774a72cd9') nome='Cauã Silva' telefone='62788888888' email='caua5562@gmail.com' senha=SecretStr('**********') cargo=<Cargo.Cliente: 0>
vendedor_1
id=UUID('77fad306-4b3a-4d06-bb9a-76f8b6b21745') nome='Pedro Augusto' telefone='11988887777' email='pedro.augusto@gmail.com' senha=SecretStr('**********') cargo=<Cargo.Vendedor: 1>
-------------------------  CARROS  -------------------------
carro_bom_1
id=UUID('18e2b4c3-42aa-46cf-850c-00c29eac9d2a') marca='Fiat' modelo='Onix' ano=2020 preco=65000.0 cor='Preto' vendedor_id=UUID('31c1e049-c6f6-495a-9b00-43333b52ee95')
carro_bom_2
id=UUID('467011aa-7736-4a94-a4f3-1b87d4f297eb') marca='Toyota' modelo='Corolla' ano=2022 preco=130000.0 cor='Branco'

Uma coisa interessante que eu descobri nesse estudo, a implementação usada muta, quando em dicionários simples, a senha para uma str criptografada em hex, o que impede quaisquer correções futuras, isso poderia ser evitado ao fazer deepcopy() do dict ou criptografar a senha em um passo futuro, o que, dependendo da implementação, poderia reduzir um pouco a segurança.  
Fora isso foi interessante utilizar Pydantic para garantir os tipos corretos, mesmo já tendo o exemplo do vídeo esse código não saiu de primeira por causa que o Pydantic encontrou erros que eu deixei passar.